# Inspect IAEVALL03 dataset

This notebook loads the dataset configuration, discovers variable files, and opens an example variable dataset with xarray.

In [2]:
from pathlib import Path
import sys
import importlib

repo_root = Path.cwd().resolve()
for candidate in [repo_root, *repo_root.parents]:
    if (candidate / 'src' / 'my_ml_zoo').exists():
        repo_root = candidate
        break
else:
    raise RuntimeError('Unable to locate repository root containing src/my_ml_zoo')

src_path = repo_root / 'src'
if str(src_path) not in sys.path:
    sys.path.insert(0, str(src_path))

importlib.invalidate_caches()
for module_name in [
    'my_ml_zoo.data',
    'my_ml_zoo',
    'my_ml_zoo.data.dataset_config',
    'my_ml_zoo.data.file_discovery',
    'my_ml_zoo.data.xarray_dataset',
]:
    if module_name in sys.modules:
        del sys.modules[module_name]

from my_ml_zoo.data import (
    DatasetConfig,
    discover_variable_files,
    select_variable_file,
    open_variable_dataset,
    infer_file_year,
)


In [3]:
config_path = repo_root / 'configs' / 'datasets' / 'IAEVALL03.json'
dataset_config = DatasetConfig.load_from_json(config_path)

print('Dataset name:', dataset_config.dataset_name)
print('Description:', dataset_config.description)
print('Base directory:', dataset_config.base_directory)
print('File format:', dataset_config.file_format)
print('Storage layout:', dataset_config.storage_layout)
print('Variables:', len(dataset_config.get_variable_names()))
print(dataset_config.get_variable_names())


Dataset name: IAEVALL03
Description: ERA5-driven historical ICON-CLM regional climate model run over Europe
Base directory: /work/bb1364/production_runs/work/IAEVALL03/yearly
File format: ncz
Storage layout: StorageLayout(variables_in_subdirectories=True, default_file_pattern='*.ncz')
Variables: 11
['T_2M', 'TOT_PREC', 'ASOD_S', 'ACLCT', 'U_10M', 'V_10M', 'PMSL', 'HSURF', 'FR_LAND', 'HPBL', 'RELHUM_2M']


In [4]:
for variable_name in dataset_config.get_variable_names():
    variable_config = dataset_config.get_variable_config(variable_name)
    paths = discover_variable_files(dataset_config, variable_name)
    print(f'{variable_name}: {len(paths)} files')
    print('  temporal resolution:', variable_config.native_temporal_resolution)
    print('  sample:', paths[:5])
    if paths:
        if not variable_config.is_time_series:
            print('  static variable: no time coordinate expected')
        try:
            inferred_year = infer_file_year(paths[0])
            print('  inferred sample file year:', inferred_year)
        except Exception as exc:
            print('  year inference skipped:', exc)


T_2M: 75 files
  temporal resolution: hourly
  sample: [PosixPath('/work/bb1364/production_runs/work/IAEVALL03/yearly/T_2M/T_2M_1950010100-1950123123.ncz'), PosixPath('/work/bb1364/production_runs/work/IAEVALL03/yearly/T_2M/T_2M_1951010100-1951123123.ncz'), PosixPath('/work/bb1364/production_runs/work/IAEVALL03/yearly/T_2M/T_2M_1952010100-1952123123.ncz'), PosixPath('/work/bb1364/production_runs/work/IAEVALL03/yearly/T_2M/T_2M_1953010100-1953123123.ncz'), PosixPath('/work/bb1364/production_runs/work/IAEVALL03/yearly/T_2M/T_2M_1954010100-1954123123.ncz')]
  inferred sample file year: 1950
TOT_PREC: 75 files
  temporal resolution: hourly
  sample: [PosixPath('/work/bb1364/production_runs/work/IAEVALL03/yearly/TOT_PREC/TOT_PREC_1950010100-1951010100.ncz'), PosixPath('/work/bb1364/production_runs/work/IAEVALL03/yearly/TOT_PREC/TOT_PREC_1951010100-1952010100.ncz'), PosixPath('/work/bb1364/production_runs/work/IAEVALL03/yearly/TOT_PREC/TOT_PREC_1952010100-1953010100.ncz'), PosixPath('/work/b

In [5]:
print('Opening T_2M dataset with xarray for a single year...')
selected_file = select_variable_file(dataset_config, 'T_2M', year=2000)
print('Selected file:', selected_file.name)

ds_t2m = open_variable_dataset(dataset_config, 'T_2M', year=2000)
ds_t2m

Opening T_2M dataset with xarray for a single year...
Selected file: T_2M_2000010100-2000123123.ncz


<xarray.Dataset>
Dimensions:       (time: 8784, height: 1, rlat: 412, rlon: 424, nv: 4)
Coordinates:
  * height        (height) float64 2.0
    lat           (rlat, rlon) float32 ...
    lon           (rlat, rlon) float32 ...
  * rlat          (rlat) float32 -23.38 -23.26 -23.16 ... 21.61 21.73 21.83
  * rlon          (rlon) float32 -28.38 -28.26 -28.16 ... 17.93 18.05 18.16
  * time          (time) datetime64[ns] 2000-01-01 ... 2000-12-31T23:00:00
Dimensions without coordinates: nv
Data variables:
    T_2M          (time, height, rlat, rlon) float32 ...
    lat_bnds      (rlat, rlon, nv) float32 ...
    lon_bnds      (rlat, rlon, nv) float32 ...
    rotated_pole  |S1 ...
Attributes: (12/16)
    CDI:               Climate Data Interface version 2.4.0 (https://mpimet.m...
    Conventions:       CF-1.4
    source:            https://gitlab.dkrz.de/icon/icon-model.git@7016082e7bf...
    institution:       CLMcom-ETH
    title:             EURO-CORDEX 0.11 simulation with ICON-CLM and SPICE 2.3
    history:           Wed Jun 11 05:13:49 2025: cdo -s -P 1 remap,/capstor/s...
    ...                ...
    realization:       1
    ConventionsURL:    http://www.cfconventions.org/
    contact:           support@c2sm.ethz.ch
    icon-clm_version:  ICON-2024.07
    creation_date:     2025-06-11 04:42:14 CEST
    CDO:               Climate Data Operators version 2.4.0 (https://mpimet.m...

In [6]:
print('Opening HSURF static variable ...')
selected_file = select_variable_file(dataset_config, 'HSURF')
print('HSURF file:', selected_file.name)
ds_hsurf = open_variable_dataset(dataset_config, 'HSURF')
ds_hsurf

Opening HSURF static variable ...
HSURF file: HSURF.nc


<xarray.Dataset>
Dimensions:       (rlat: 412, rlon: 424, nv: 4)
Coordinates:
    lat           (rlat, rlon) float32 ...
    lon           (rlat, rlon) float32 ...
  * rlat          (rlat) float32 -23.38 -23.26 -23.16 ... 21.61 21.73 21.83
  * rlon          (rlon) float32 -28.38 -28.26 -28.16 ... 17.93 18.05 18.16
Dimensions without coordinates: nv
Data variables:
    HSURF         (rlat, rlon) float32 ...
    lat_bnds      (rlat, rlon, nv) float32 ...
    lon_bnds      (rlat, rlon, nv) float32 ...
    rotated_pole  |S1 ...
Attributes: (12/20)
    CDI:                        Climate Data Interface version 2.4.0 (https:/...
    Conventions:                CF-1.4
    uuidOfVGrid:                6da16f93-6445-ed9e-92a2-623f9e722ca0
    source:                     https://gitlab.dkrz.de/icon/icon-model.git@8e...
    institution:                CLMcom-ETH
    references:                 see MPIM/DWD publications
    ...                         ...
    creation_date:              2025-12-03 16:42:47 CET
    NCO:                        netCDF Operators version 5.1.9 (Homepage = ht...
    history:                    Wed Dec 03 16:42:51 2025: cdo -s setmissval,-...
    history_of_appended_files:  Wed Dec  3 16:42:49 2025: Appended file /caps...
    rawdata:                    GLOBCOVER2009, HWSD2.0, MERIT, Lake Database
    CDO:                        Climate Data Operators version 2.4.0 (https:/...

## Some checks for future normalization

In [21]:
ds_t2m_2000 = open_variable_dataset(dataset_config, 'T_2M', year=2000)['T_2M']
ds_t2m_2000 = ds_t2m_2000.squeeze('height')
print(ds_t2m_2000.dims)
print(ds_t2m_2000.shape)
print(ds_t2m_2000.dtype)
print("min, max:", ds_t2m_2000.min().item(),", ",ds_t2m_2000.max().item())
print(ds_t2m_2000.isnull().sum().item())

('time', 'rlat', 'rlon')
(8784, 412, 424)
float32
min, max: 225.0616912841797 ,  324.0158386230469
0
